# 01 — Data audit (Phase 1)

Every prepared layer gets a map, counts per district and a missing-value check
(README §7, Phase 1). By default this runs **offline on the committed sample**
(`config/config.sample.yaml`). To audit the full city after running
`almaty-access download && almaty-access prepare`, point `CONFIG_PATH` at
`config/config.yaml` instead.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio

from almaty_access.config import load_config
from almaty_access.io import read_geoparquet

# Default: the offline sample. To audit the full city instead, set the
# environment variable ALMATY_CONFIG=config/config.yaml before starting
# Jupyter (or edit the fallback below).
override = os.environ.get("ALMATY_CONFIG")
candidates = (
    [Path(override), Path("..") / override]
    if override
    else [Path("config/config.sample.yaml"), Path("../config/config.sample.yaml")]
)
CONFIG_PATH = next(p for p in candidates if p.exists())
config = load_config(CONFIG_PATH)
print("config :", CONFIG_PATH.resolve())
print("raw    :", config.paths.raw)
print("interim:", config.paths.interim)

If the interim layers do not exist yet, build them now (cached, offline,
a few seconds on the sample).

In [ ]:
from almaty_access.prepare.boundary import prepare_boundary
from almaty_access.prepare.buildings import prepare_buildings
from almaty_access.prepare.completeness import prepare_completeness
from almaty_access.prepare.dem import prepare_dem
from almaty_access.prepare.network import prepare_networks
from almaty_access.prepare.pois import prepare_pois
from almaty_access.prepare.population import prepare_population

prepare_boundary(config)
prepare_dem(config)
prepare_pois(config)
prepare_buildings(config)
prepare_population(config)
prepare_networks(config)
prepare_completeness(config)
print("interim layers ready")

## Boundary: city, districts, buffer

In [ ]:
city = read_geoparquet(config.paths.interim / "boundary_city.parquet")
districts = read_geoparquet(config.paths.interim / "boundary_districts.parquet")
buffer = read_geoparquet(config.paths.interim / "boundary_buffer.parquet")

fig, ax = plt.subplots(figsize=(7, 7))
buffer.plot(ax=ax, color="#eef2f7", edgecolor="#9fb3c8", linewidth=0.8)
districts.plot(ax=ax, column="district", cmap="tab20", alpha=0.6, edgecolor="white")
city.boundary.plot(ax=ax, color="black", linewidth=1.2)
ax.set_title("City boundary, 8 districts, 5 km buffer")
ax.set_axis_off()
plt.show()

districts[["district", "name_osm", "area_km2"]].sort_values("area_km2", ascending=False)

## Buildings: sources, functions, floors, GFA

In [ ]:
buildings = read_geoparquet(config.paths.interim / "buildings.parquet")
print(f"{len(buildings):,} footprints | total GFA {buildings.gfa_m2.sum() / 1e6:.2f} km²")
display(pd.crosstab(buildings["function"], buildings["src"], margins=True))
display(buildings.groupby("levels_source")["levels"].describe().round(2))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
buildings.plot(ax=axes[0], column="function", legend=True, cmap="Set2", linewidth=0)
axes[0].set_title("Building function")
buildings.plot(
    ax=axes[1],
    column="gfa_m2",
    scheme=None,
    cmap="viridis",
    linewidth=0,
    norm=plt.matplotlib.colors.LogNorm(
        vmin=max(buildings.gfa_m2.min(), 10), vmax=buildings.gfa_m2.max()
    ),
)
axes[1].set_title("GFA per building (log scale)")
for ax in axes:
    ax.set_axis_off()
plt.show()

## POIs by category

In [ ]:
pois = read_geoparquet(config.paths.interim / "pois.parquet")
counts = pois["category"].value_counts()
print(f"{len(pois):,} POIs in {len(counts)} categories")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
counts.sort_values().plot.barh(ax=axes[0], color="#4878a8")
axes[0].set_title("POI count per category")
top = counts.head(8).index
pois[pois.category.isin(top)].plot(
    ax=axes[1], column="category", legend=True, markersize=6, cmap="tab10"
)
axes[1].set_title("POIs (top 8 categories)")
axes[1].set_axis_off()
plt.tight_layout()
plt.show()

## Population: resolution 8 and the dasymetric resolution 9

The r9 layer must conserve the r8 total exactly (mass-preserving split by
residential GFA).

In [ ]:
pop8 = read_geoparquet(config.paths.interim / "population_r8.parquet")
pop9 = read_geoparquet(config.paths.interim / "population_r9.parquet")
print(f"r8: {len(pop8):,} hexes, {pop8.population.sum():,.0f} residents")
print(f"r9: {len(pop9):,} hexes, {pop9.population.sum():,.0f} residents")
assert np.isclose(pop8.population.sum(), pop9.population.sum()), "r9 must conserve the r8 total"

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
pop8.plot(
    ax=axes[0], column="population", legend=True, cmap="magma", edgecolor="white", linewidth=0.3
)
axes[0].set_title("Population, H3 r8 (Kontur)")
pop9.plot(ax=axes[1], column="population", legend=True, cmap="magma", linewidth=0)
axes[1].set_title("Population, H3 r9 (dasymetric by residential GFA)")
for ax in axes:
    ax.set_axis_off()
plt.show()

## Terrain: elevation and slope (metric CRS)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, name, cmap in ((axes[0], "dem_32643.tif", "terrain"), (axes[1], "slope_32643.tif", "OrRd")):
    with rasterio.open(config.paths.interim / name) as src:
        band = src.read(1).astype(float)
        band[band == src.nodata] = np.nan
    im = ax.imshow(band, cmap=cmap)
    fig.colorbar(im, ax=ax, shrink=0.8)
    ax.set_title(name)
    ax.set_axis_off()
plt.show()

## Networks: size, length, grades

In [ ]:
rows = []
for mode in ("walk", "bike", "drive"):
    nodes = read_geoparquet(config.paths.interim / f"network_{mode}_nodes.parquet")
    edges = read_geoparquet(config.paths.interim / f"network_{mode}_edges.parquet")
    rows.append(
        {
            "mode": mode,
            "nodes": len(nodes),
            "edges": len(edges),
            "length_km": edges.length_m.sum() / 1000,
            "mean_abs_grade": float(edges.grade.abs().mean()),
            "nodes_without_elevation": int(nodes.elevation_m.isna().sum()),
        }
    )
display(pd.DataFrame(rows).round(3))

walk_edges = read_geoparquet(config.paths.interim / "network_walk_edges.parquet")
ax = walk_edges.plot(
    figsize=(7, 7), linewidth=0.4, column=walk_edges.grade.abs(), cmap="OrRd", legend=True
)
ax.set_title("Walk network, |grade|")
ax.set_axis_off()
plt.show()

## OSM completeness vs Overture, and missing values

In [ ]:
completeness = pd.read_csv(config.paths.interim / "completeness.csv")
display(completeness)

print("Missing-value summary (share of NA per column):")
for name, frame in (("buildings", buildings), ("pois", pois), ("population_r8", pop8)):
    na = frame.drop(columns=frame.geometry.name).isna().mean().round(3)
    na = na[na > 0]
    print(f"\n[{name}]")
    print(na.to_string() if len(na) else "  no missing values")

## Audit verdict

- Every interim layer loads, has a CRS and plots where Almaty is.
- The r9 population conserves the r8 total (asserted above).
- District stats: see `data/raw/district_stats.csv` — two indicators with
  very different meanings (employed *residents* vs payroll *jobs located*);
  the jobs series is the Phase 3 target.
- Districts flagged `weak_osm_coverage` need caution when interpreting
  model outputs there (features under-observed in OSM).